# The integrity gate that vouches for the corpus, and the controls that make the engine refuse

**Level 4: Beyond the state of the art** · Needs: 01-the-data/01-the-corpus-and-the-cases, 01-the-data/02-geometry-reconstruction · Reading time: about 25 minutes

## Objectives

A table copied from a paper is only as good as the copy. This notebook opens the two safeguards that Fragmenta puts around its data: a gate that reproduces the paper's own printed statistics from the shipped rows every time the corpus loads (claim BS-3), and a pair of negative controls that prove the engine refuses where a number would be an invention (claim BS-2).

By the end you will have:

- recomputed the minimum, maximum, mean and standard deviation of all seven inputs and compared them with the values the paper prints;
- planted four realistic transcription defects in a copy of the corpus and seen which of the gate's two layers catches each one;
- seen the geometry negative control (six blasts, no scale) and the degenerate-design negative control (six designs that are not blasts) pass.

## Setup

Install the packages with the exact pins of `notebooks/requirements.txt`. The engine is `blastfrag`, published on PyPI; the other packages are its numerical base and the plotting library.

```bash
pip install blastfrag==0.4.0 numpy==2.5.4
```

This cell imports nothing from the notebook yet: it reads the installed versions from the package metadata and prints them, so the output records exactly which engine produced every number below.

In [1]:
# Print the installed version of every package this notebook uses
from importlib.metadata import version

for name in ("blastfrag", "numpy",):
    print(f"{name}=={version(name)}")

blastfrag==0.4.0
numpy==2.5.4


## Data

| Source | Owner and URL | Licence class | File the notebook reads |
|---|---|---|---|
| The 97 rows and the printed descriptive statistics | Hudaverdi, Kulatilake and Kuzu 2010, https://doi.org/10.1002/nag.957 | numeric facts reused with citation; the article never ships | the table and `PUBLISHED_DESCRIPTIVE_STATS` inside the `blastfrag` wheel |
| The geometry negative control (Miami) | this repository (MIT), written by the offline bake | the product's own licence | `data/derived/real-miami/case.json` |
| The degenerate-design negative control | this repository (MIT), written by the offline bake | the product's own licence | `data/derived/ctrl-degenerate/case.json` |

## Method

The gate has two independent layers, and they answer different questions.

1. **The printed statistics.** The paper prints a minimum, a maximum, a mean and a standard deviation for each of the seven inputs. The gate recomputes all four from the shipped rows and compares each with the printed value *at the precision it was printed to*. A mismatch means the file is no longer the published table.
2. **The content digest.** The statistics are a weak detector of a single changed cell: correcting one powder factor moves a mean over 97 rows by less than a rounding step. A SHA-256 digest of the whole table, pinned in the engine, catches any change at all.

Both run inside `load_training_corpus()`. The first cell calls the gate directly and lays the recomputed statistics beside the printed ones.

In [2]:
# Imports, then the gate on the shipped corpus, with the recomputed statistics beside the printed ones
import dataclasses
import json
import math
from pathlib import Path

import blastfrag as bf

DERIVED = Path("../../../data/derived")
train = bf.load_training_corpus()
bf.check_source_integrity(train)
print("gate on the shipped corpus: passed")

print(f"{'input':<10}{'statistic':<14}{'paper prints':>16}{'recomputed':>14}")
for name, printed in bf.PUBLISHED_DESCRIPTIVE_STATS.items():
    values = [getattr(b, name) for b in train]
    mean = sum(values) / len(values)
    sd = math.sqrt(sum((v - mean) ** 2 for v in values) / (len(values) - 1))
    for label, want, got in (("min", printed.minimum, min(values)), ("max", printed.maximum, max(values)),
                              ("mean", printed.mean, mean), ("sd", printed.sd, sd)):
        print(f"{name:<10}{label:<14}{want:>16}{got:>14.4f}")

gate on the shipped corpus: passed
input     statistic         paper prints    recomputed
S_over_B  min                        1.0        1.0000
S_over_B  max                       1.75        1.7500
S_over_B  mean                      1.19        1.1890
S_over_B  sd                       0.117        0.1168
H_over_B  min                       1.33        1.3300
H_over_B  max                       6.82        6.8200
H_over_B  mean                      3.34        3.3452
H_over_B  sd                       1.634        1.6337
B_over_D  min                      17.98       17.9800
B_over_D  max                      39.47       39.4700
B_over_D  mean                     27.35       27.3549
B_over_D  sd                       4.838        4.8386
T_over_B  min                        0.5        0.5000
T_over_B  max                       4.67        4.6700
T_over_B  mean                      1.26        1.2628
T_over_B  sd                       0.674        0.6739
Pf_kg_m3  min                 

## State of the art

Three facts from the product's research basis (the SDD) frame what is already known.

- The corpus is a published table. Hudaverdi, Kulatilake and Kuzu 2010 (https://doi.org/10.1002/nag.957) print it together with its own summary statistics, which is what makes an independent check possible at all.
- The SDD research basis records that a copy of the table held in the lab's vault carried five transcription defects, two of them on the regression target, found by comparing every cell with the paper's tables. A study that reuses a copied table without that comparison inherits such defects silently.
- The studies that fit learned predictors to these rows (Amoako, Jha and Zhong 2022, https://doi.org/10.3390/mining2020013; Sui et al. 2025, https://doi.org/10.3390/app15031254) report predictors of the seven inputs. A trained regressor returns a number for any input vector; it has no declared domain in which it declines to answer. Whether the rock volume and charge can be known for a blast is outside what such a predictor can express.

## Beyond the state of the art

**Claim BS-3, the load-time integrity gate.** Every load of the corpus reproduces the paper's printed statistics and a pinned digest. The cell below plants four defects in copies of the corpus and runs the gate on each, first with the statistics only (`check_digest=False`) and then with both layers. The first defect is the historical one the gate was written after: a powder-factor maximum of 1.47 where the paper prints 1.26.

In [3]:
# Plant one transcription defect at a time in a copy of the corpus and run both layers of the gate on it
peak = max(train, key=lambda b: b.Pf_kg_m3)
first = train[0]
defects = {
    "powder-factor maximum 1.26 typed as 1.47": (peak.blast_id, {"Pf_kg_m3": 1.47}),
    "modulus 60.0 typed as 6.0": (first.blast_id, {"E_GPa": 6.0}),
    "stemming ratio 0.78 typed as 0.79": (first.blast_id, {"T_over_B": 0.79}),
    "measured size 0.37 typed as 0.73": (first.blast_id, {"x50_m": 0.73}),
}

def verdict(rows, with_digest):
    # True when the gate raises, which is the outcome we want for a planted defect
    try:
        bf.check_source_integrity(rows, check_digest=with_digest)
        return False
    except bf.ContractViolation:
        return True

print(f"{'planted defect':<62}{'statistics':>14}{'statistics + digest':>26}")
for label, (blast_id, change) in defects.items():
    corrupted = [dataclasses.replace(b, **change) if b.blast_id == blast_id else b for b in train]
    print(f"{label:<62}{'caught' if verdict(corrupted, False) else 'missed':>14}"
          f"{'caught' if verdict(corrupted, True) else 'missed':>26}")

planted defect                                                    statistics       statistics + digest
powder-factor maximum 1.26 typed as 1.47                              caught                    caught
modulus 60.0 typed as 6.0                                             caught                    caught
stemming ratio 0.78 typed as 0.79                                     missed                    caught
measured size 0.37 typed as 0.73                                      missed                    caught


**Claim BS-2, abstention as a first-class result.** An arm that needs a rock volume must not answer where none exists. The geometry control is the six Miami blasts. The cell reads the committed artifact of that case and counts, for each of the 14 arms, how many of the six blasts it answered and how many it refused. The classical arms must refuse all six; the arms that use only the ratios may answer.

In [4]:
# Count answered and refused blasts per arm in the committed Miami artifact
miami = json.loads((DERIVED / "real-miami" / "case.json").read_text(encoding="utf-8"))
print(f"{'arm':<24}{'answered':>10}{'refused':>10}")
for arm, per_blast in sorted(miami["predictions"].items()):
    refused = sum(1 for p in per_blast.values() if p["abstained"])
    print(f"{arm:<24}{len(per_blast) - refused:>10}{refused:>10}")
print("control recorded by the bake:", miami["controls"]["geometry_negative_control"])

# The same refusal, live from the installed engine
blast = next(b for b in train if b.site == "Miami")
live = bf.Kuznetsov().predict_one(blast)
print("live engine, Kuznetsov, blast", blast.blast_id, "->", live.x50_m, "|", live.abstain_reason[:60], "...")

arm                       answered   refused
crush-zone                       0         6
group-discriminant               0         6
kuz-ram                          0         6
kuznetsov                        0         6
kuznetsov-capped                 0         6
kuznetsov-transfer               0         6
published-neural-net             6         0
published-regression             6         0
random-forest                    6         0
refitted-regression              6         0
stacking                         6         0
svr-rbf                          6         0
swebrec                          0         6
xgboost                          6         0
control recorded by the bake: {'n_answered_anyway': 0, 'n_without_geometry': 6, 'passed': True}
live engine, Kuznetsov, blast Mi1 -> None | Miami publishes no hole diameter, so rock volume and charge  ...


The second control is a set of six designs that are not blasts: stemming longer than the bench, so there is no charge column. A ratio-only model would happily price them. The product guards the design before any arm runs, so every arm must refuse every one.

In [5]:
# For every arm count the degenerate designs it answered; the control passes when the total is zero
degenerate = json.loads((DERIVED / "ctrl-degenerate" / "case.json").read_text(encoding="utf-8"))
answered = {
    arm: sum(1 for p in per_blast.values() if not p["abstained"])
    for arm, per_blast in degenerate["predictions"].items()
}
print("arms checked:", len(answered), "| designs answered by any arm:", sum(answered.values()))
print("control recorded by the bake:", degenerate["controls"]["degenerate_negative_control"])
first_design = degenerate["blasts"][0]
print(first_design["blast_id"], "reason (en):", first_design["degenerate_reason"]["en"])
print(first_design["blast_id"], "reason (es):", first_design["degenerate_reason"]["es"])

arms checked: 14 | designs answered by any arm: 0
control recorded by the bake: {'answered_by': [], 'n_answered_anyway': 0, 'n_degenerate': 6, 'passed': True}
DG01 reason (en): the stemming is 4.60 burdens in a bench 1.40 burdens tall, so there is no charge column. This is not a blast.
DG01 reason (es): el taco mide 4,60 bordos en un banco de 1,40 bordos de alto, así que no hay columna de carga. Esto no es una voladura.


## Results

**The gate holds the corpus to the paper.** On the shipped rows the recomputed minimum, maximum, mean and standard deviation of all seven inputs agree with the printed values at the precision the paper used, and the digest matches.

**The two layers are not redundant.** The powder-factor maximum typed as 1.47 and the modulus typed as 6.0 move a printed extreme, and the statistics layer catches them. The stemming ratio changed in the second decimal of one row, and the measured size transposed, move no printed statistic by a visible amount (the measured size is not among the printed statistics at all), so only the digest catches them. The notebook shows each outcome in the table above rather than asserting it. A copied table checked only against summary statistics would let such a defect through.

**The refusals work as controls.** In the Miami artifact the arms that need a rock volume answer none of the six blasts, while the arms that use only the ratios answer all six (the group discriminant returns a rock-stiffness group, not a size, so it appears as refusing on every row): the engine does not hide the difference, it reports it. In the degenerate case no arm answers any of the six designs, and each refusal explains itself in English and Spanish.

**What this does not show.** Passing the gate shows the file is the published table; it does not show the published table is right about the blasts. And a refusal is not a proof of correctness elsewhere: the controls show the engine declines where it must, not that it is accurate where it answers. That question is the subject of topics 04 and 05.

## Summary

- Loading the corpus reproduces the paper's 28 printed statistics (4 for each of 7 inputs) and a pinned digest; the statistics catch changes to an extreme and the digest catches the rest (claim BS-3).
- Two negative controls ship as data: six blasts with no scale and six designs that are not blasts. The engine refuses all twelve where it must, with reasons in both languages (claim BS-2).
- Next: topic 02 uses the reconstructed geometry to run the classical equation.

## References

- Hudaverdi, Kulatilake and Kuzu 2010, https://doi.org/10.1002/nag.957 (the table, its printed statistics, the five defects found by comparison)
- Amoako, Jha and Zhong 2022, https://doi.org/10.3390/mining2020013 (learned predictors on the same rows)
- Sui, Zhou, Zhao, Yang and Zou 2025, https://doi.org/10.3390/app15031254 (the stacking ensemble and the field blasts)
- `blastfrag` 0.4.0, the engine, on PyPI: https://pypi.org/project/blastfrag/